In [ ]:
# Parámetros (fijados ANTES de comparar fuentes)
CUTOFF_DAY = 31
MIN_ARTICLES = 30
OVERLAP_PRIMARY = ("2015-01", "2016-12")
OVERLAP_SECONDARY = ("2015-01", "2020-06")
THRESHOLD_CORR = 0.7
USE_UNION_VARIANT = True  # variantes unión por decisión explícita, no por la regla

SEED = 42
SANITY_N = 25          # titulares por fuente para la revisión manual

# Model 1 — Monthly news index (InflaBERT, headlines only)

**Threshold fixed BEFORE comparing sources.** `THRESHOLD_CORR`, the overlap
windows, `CUTOFF_DAY` and `MIN_ARTICLES` are set in the first cell and must not be
changed after looking at section 4.

**CUTOFF_DAY=31: index of month t uses the whole month t, available before the CPI release of month t.**

Inputs: `data/processed/news_scored_alpaca.parquet` and
`news_scored_fnspid.parquet` (headline-only InflaBERT scores).
This notebook loads **no CPI and no target**, and never prints headlines.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Raíz del repo: primer directorio hacia arriba con pyproject.toml
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.model1.aggregate import aggregate_monthly

PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)
SOURCES = ["alpaca", "fnspid"]
INDEX_COLS = ["month", "infla_score_mean", "infla_share_up",
              "infla_share_down", "infla_net_share", "news_n_articles"]

# Paleta categórica de referencia (slots 1 y 2), tinta y superficie
COLORS = {"alpaca": "#2a78d6", "fnspid": "#eb6834"}
INK, INK_2, GRID, SURFACE = "#1f1f1e", "#5f5e58", "#e4e3df", "#fcfcfb"
pd.set_option("display.width", 200)
print("repo root:", ROOT)

## 1. Load both sources
Rows, date range and the share of each InflaBERT role (up / down / neu) per year.

In [ ]:
scored = {s: pd.read_parquet(PROCESSED / f"news_scored_{s}.parquet")
          for s in SOURCES}

summary = pd.DataFrame([
    {"source": s, "rows": len(d), "first": d["Date"].min(),
     "last": d["Date"].max(), "n_domains": d["domain"].nunique()}
    for s, d in scored.items()
])
summary

In [ ]:
# Share de cada rol por año y fuente (filas = año)
role_tables = {}
for s, d in scored.items():
    t = pd.crosstab(d["Date"].dt.year, d["infla_role"], normalize="index")
    t = t.reindex(columns=["up", "down", "neu"], fill_value=0.0)
    t.insert(0, "n", d["Date"].dt.year.value_counts().sort_index())
    t.index.name = "year"
    role_tables[s] = t
    print(f"\n{s}: overall shares", (d["infla_role"].value_counts(normalize=True)
                                       .round(3).to_dict()))
    display(t.style.format({"up": "{:.1%}", "down": "{:.1%}", "neu": "{:.1%}"}))

## 2. Sanity sample for human labeling
25 random headlines per source (seed 42) are written to
`reports/sanity_sample_news_index.csv` with an empty `human_price_direction`
column. **The headlines are not printed here.**

In [ ]:
samples = []
for s, d in scored.items():
    smp = d.sample(n=SANITY_N, random_state=SEED)
    samples.append(pd.DataFrame({
        "source": s,
        "Date": smp["Date"],
        "title": smp["Article_title"],
        "infla_role": smp["infla_role"],
        "infla_score": smp["infla_score"],
        "human_price_direction": "",
    }))
sanity = pd.concat(samples, ignore_index=True)
sanity_path = REPORTS / "sanity_sample_news_index.csv"
sanity.to_csv(sanity_path, index=False)
# Solo conteos: no se imprimen titulares
print(f"wrote {sanity_path} ({len(sanity)} rows)")
print(sanity.groupby(["source", "infla_role"]).size().unstack(fill_value=0))

## 3. Monthly index per source
`aggregate_monthly(scored, CUTOFF_DAY)` from `src/model1/aggregate.py`: only
articles from UTC days 1..CUTOFF_DAY count for month t. Months with fewer than
`MIN_ARTICLES` articles are flagged (`below_min_flag`), not dropped.

In [ ]:
index = {}
for s, d in scored.items():
    m = aggregate_monthly(d, CUTOFF_DAY)[INDEX_COLS].copy()
    m["below_min_flag"] = m["news_n_articles"] < MIN_ARTICLES
    m["source"] = s
    index[s] = m
    m.to_csv(PROCESSED / f"news_index_{s}.csv", index=False)
    print(f"wrote {PROCESSED / f'news_index_{s}.csv'} ({len(m)} months)")


def month_runs(months):
    """Agrupa meses consecutivos en rangos 'YYYY-MM..YYYY-MM'."""
    p = pd.PeriodIndex(pd.to_datetime(months), freq="M")
    runs, start, prev = [], None, None
    for x in p:
        if start is None:
            start = prev = x
        elif x == prev + 1:
            prev = x
        else:
            runs.append((start, prev)); start = prev = x
    if start is not None:
        runs.append((start, prev))
    return [f"{a}" if a == b else f"{a}..{b}" for a, b in runs]


for s, m in index.items():
    low = m.loc[m["below_min_flag"], "month"]
    print(f"\n{s}: {m['month'].min():%Y-%m}..{m['month'].max():%Y-%m}, "
          f"{len(m)} months; below {MIN_ARTICLES}: {len(low)}")
    print("  ", ", ".join(month_runs(low)) if len(low) else "  none")

In [ ]:
# Resumen por año: artículos en ventana y score medio (solo meses >= MIN_ARTICLES)
rows = []
for s, m in index.items():
    ok = m[~m["below_min_flag"]]
    g = m.groupby(m["month"].dt.year)
    rows.append(pd.DataFrame({
        "source": s,
        "months": g.size(),
        "articles_min": g["news_n_articles"].min(),
        "articles_median": g["news_n_articles"].median(),
        "months_below_min": g["below_min_flag"].sum(),
        "score_mean": ok.groupby(ok["month"].dt.year)["infla_score_mean"].mean(),
        "net_share_mean": ok.groupby(ok["month"].dt.year)["infla_net_share"].mean(),
    }))
by_year = pd.concat(rows).rename_axis("year").reset_index()
by_year.pivot(index="year", columns="source",
              values=["articles_median", "months_below_min", "score_mean"]).round(3)

In [ ]:
# Dos paneles (sin doble eje): score medio arriba, artículos por mes abajo (log)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                               height_ratios=[3, 2], facecolor=SURFACE)
for s, m in index.items():
    ok = m["news_n_articles"] >= MIN_ARTICLES
    score = m["infla_score_mean"].where(ok)   # meses con pocos artículos: hueco
    ax1.plot(m["month"], score, color=COLORS[s], lw=1.8, label=s)
    ax2.plot(m["month"], m["news_n_articles"].replace(0, np.nan),
             color=COLORS[s], lw=1.8, label=s)
    # Etiqueta directa al final de cada serie
    last = m.loc[ok].iloc[-1]
    ax1.annotate(s, (last["month"], last["infla_score_mean"]),
                 xytext=(6, 0), textcoords="offset points",
                 color=INK, fontsize=9, va="center")

ax1.axhline(0, color=INK_2, lw=0.8)
ax2.axhline(MIN_ARTICLES, color=INK_2, lw=1, ls="--")
# Eje x desde 2007-01: FNSPID 2000-2006 tiene <= 3 artículos por mes
X_START = pd.Timestamp("2007-01-01")
ax2.annotate(f"MIN_ARTICLES = {MIN_ARTICLES}", (X_START, MIN_ARTICLES),
             xytext=(4, 4), textcoords="offset points", color=INK_2, fontsize=8)
ax2.set_yscale("log")
ax1.set_xlim(left=X_START)
for a0, a1 in [OVERLAP_PRIMARY]:
    for ax in (ax1, ax2):
        ax.axvspan(pd.Timestamp(a0), pd.Timestamp(a1) + pd.offsets.MonthEnd(),
                   color=GRID, alpha=0.6, lw=0)
ax1.set_title(f"InflaBERT infla_score_mean by month (days 1–{CUTOFF_DAY}; "
              f"months with < {MIN_ARTICLES} articles hidden; shaded = "
              f"primary overlap; x from 2007-01)", color=INK, fontsize=11,
              loc="left")
ax1.set_ylabel("infla_score_mean", color=INK_2)
ax2.set_ylabel("articles per month (log)", color=INK_2)
for ax in (ax1, ax2):
    ax.set_facecolor(SURFACE)
    ax.grid(axis="y", color=GRID, lw=0.8)
    ax.tick_params(colors=INK_2, labelsize=9)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
ax1.legend(frameon=False, loc="upper left", labelcolor=INK)
fig.tight_layout()
plot_path = REPORTS / "news_index_sources.png"
fig.savefig(plot_path, dpi=130, facecolor=SURFACE)
print(f"saved {plot_path}")
plt.show()

## 4. Compare sources on the overlap and build the final index

> The decision rule uses `THRESHOLD_CORR` from the first cell and must not be
> changed after seeing these numbers.

**Rule: months under `MIN_ARTICLES` are KEPT and flagged, never dropped**, in every
file and every comparison. A month enters a comparison whenever both sources have
a score (only months with zero articles, whose score is NaN, are left out).
Pearson and Spearman of `infla_score_mean`, a Fisher-z 95% CI for Pearson r, mean
difference (FNSPID − Alpaca) and std ratio (FNSPID / Alpaca).
The secondary overlap is reported only (with flagged months included, plus one
extra line excluding them); it does not affect the decision.

In [ ]:
from scipy import stats


def overlap_frame(window, include_flagged=True):
    """Meses del rango donde ambas fuentes tienen score. Los meses marcados
    (< MIN_ARTICLES) se incluyen salvo include_flagged=False (solo reporte)."""
    a = index["alpaca"].set_index("month")
    f = index["fnspid"].set_index("month")
    lo, hi = (pd.Timestamp(window[0]), pd.Timestamp(window[1]))
    both = pd.DataFrame({
        "alpaca": a["infla_score_mean"], "fnspid": f["infla_score_mean"],
        "flag_alpaca": a["below_min_flag"], "flag_fnspid": f["below_min_flag"],
    }).loc[lo:hi].dropna(subset=["alpaca", "fnspid"])
    if not include_flagged:
        both = both[~(both["flag_alpaca"].astype(bool) | both["flag_fnspid"].astype(bool))]
    return both


def compare(window, include_flagged=True, label=""):
    o = overlap_frame(window, include_flagged)
    n = len(o)
    r, p_r = stats.pearsonr(o["fnspid"], o["alpaca"])
    rho, p_rho = stats.spearmanr(o["fnspid"], o["alpaca"])
    # IC 95% de Fisher-z para r
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    ci = np.tanh([z - 1.96 * se, z + 1.96 * se])
    n_flagged = int((o["flag_alpaca"].astype(bool) | o["flag_fnspid"].astype(bool)).sum())
    return {
        "comparison": label, "window": f"{window[0]}..{window[1]}",
        "flagged_months": "included" if include_flagged else "excluded",
        "n_months": n, "n_flagged_in_sample": n_flagged,
        "pearson_r": r, "pearson_p": p_r, "r_ci95_low": ci[0], "r_ci95_high": ci[1],
        "spearman_rho": rho, "spearman_p": p_rho,
        "mean_diff_fnspid_minus_alpaca": (o["fnspid"] - o["alpaca"]).mean(),
        "std_ratio_fnspid_over_alpaca": o["fnspid"].std() / o["alpaca"].std(),
    }


comparison = pd.DataFrame([
    compare(OVERLAP_PRIMARY, True, "primary (decision)"),
    compare(OVERLAP_SECONDARY, True, "secondary (report only)"),
    compare(OVERLAP_SECONDARY, False, "secondary, flagged excluded (report only)"),
])
comparison.to_csv(REPORTS / "news_index_source_comparison.csv", index=False)
comparison.set_index("comparison").T

In [ ]:
# Decisión (solo con la ventana PRIMARIA)
primary_r = comparison.loc[comparison["comparison"] == "primary (decision)", "pearson_r"].item()
USE_UNION = bool(primary_r >= THRESHOLD_CORR)
print(f"primary Pearson r = {primary_r:.3f} vs THRESHOLD_CORR = {THRESHOLD_CORR} "
      f"-> {'UNION FNSPID 2007-01..2014-12 + Alpaca 2015-01..2026-10' if USE_UNION else 'ALPACA ONLY'}")

FINAL_COLS = ["month", "infla_score_mean", "news_n_articles", "source",
              "below_min_flag"]


def window(df, lo, hi):
    return df[(df["month"] >= pd.Timestamp(lo)) & (df["month"] <= pd.Timestamp(hi))]


alpaca_part = window(index["alpaca"], "2015-01", "2026-10")
if USE_UNION:
    fnspid_part = window(index["fnspid"], "2007-01", "2014-12")
    final = pd.concat([fnspid_part, alpaca_part], ignore_index=True)[FINAL_COLS]

    # Versión reescalada: FNSPID llevado a la media/desvío de Alpaca usando
    # SOLO la ventana primaria de solape
    o = overlap_frame(OVERLAP_PRIMARY, include_flagged=True)
    mu_f, sd_f = o["fnspid"].mean(), o["fnspid"].std()
    mu_a, sd_a = o["alpaca"].mean(), o["alpaca"].std()
    rescaled = final.copy()
    is_f = rescaled["source"] == "fnspid"
    rescaled.loc[is_f, "infla_score_mean"] = (
        (rescaled.loc[is_f, "infla_score_mean"] - mu_f) / sd_f * sd_a + mu_a)
    rescaled.to_csv(PROCESSED / "news_index_final_rescaled.csv", index=False)
    print(f"rescale params (primary overlap): fnspid mean={mu_f:.4f} sd={sd_f:.4f}; "
          f"alpaca mean={mu_a:.4f} sd={sd_a:.4f}")
    print(f"wrote {PROCESSED / 'news_index_final_rescaled.csv'}")
else:
    final = alpaca_part[FINAL_COLS].reset_index(drop=True)
    print("Alpaca only: no FNSPID months, so no rescaled version is written.")

final.to_csv(PROCESSED / "news_index_final.csv", index=False)
print(f"wrote {PROCESSED / 'news_index_final.csv'}: {len(final)} months, "
      f"{final['month'].min():%Y-%m}..{final['month'].max():%Y-%m}, "
      f"below_min months: {int(final['below_min_flag'].sum())}")
final.groupby("source").size()

### Union variants (explicit user decision)

**Rule-based decision: Alpaca only (r=0.448 < 0.7).
Union variants below are generated by explicit user decision, not by the rule.**

`news_index_final.csv` stays rule-based. When `USE_UNION_VARIANT` is True, two extra
files are written regardless of the branch taken above:

- `news_index_union_raw.csv`: FNSPID 2007-01..2014-12 + Alpaca 2015-01..2026-10.
- `news_index_union_rescaled.csv`: same, with FNSPID rescaled to Alpaca's mean/std
  using `OVERLAP_PRIMARY` only.

Same columns as `news_index_final.csv` (including `source`); flagged months are kept.

In [ ]:
# Variantes unión: decisión explícita del usuario, NO de la regla.
# No modifican `final` ni news_index_final.csv.
if USE_UNION_VARIANT:
    union_raw = pd.concat([
        window(index["fnspid"], "2007-01", "2014-12"),
        window(index["alpaca"], "2015-01", "2026-10"),
    ], ignore_index=True)[FINAL_COLS]

    # Parámetros de reescalado SOLO de la ventana primaria (meses marcados incluidos)
    o_u = overlap_frame(OVERLAP_PRIMARY, include_flagged=True)
    mu_f_u, sd_f_u = o_u["fnspid"].mean(), o_u["fnspid"].std()
    mu_a_u, sd_a_u = o_u["alpaca"].mean(), o_u["alpaca"].std()
    union_rescaled = union_raw.copy()
    is_f_u = union_rescaled["source"] == "fnspid"
    union_rescaled.loc[is_f_u, "infla_score_mean"] = (
        (union_rescaled.loc[is_f_u, "infla_score_mean"] - mu_f_u) / sd_f_u
        * sd_a_u + mu_a_u)

    print(f"rescale params (primary overlap, n={len(o_u)}): "
          f"fnspid mean={mu_f_u:.4f} sd={sd_f_u:.4f}; "
          f"alpaca mean={mu_a_u:.4f} sd={sd_a_u:.4f}")
    for name, df in [("news_index_union_raw.csv", union_raw),
                     ("news_index_union_rescaled.csv", union_rescaled)]:
        df.to_csv(PROCESSED / name, index=False)
        print(f"wrote {PROCESSED / name}: {len(df)} rows, "
              f"{df['month'].min():%Y-%m}..{df['month'].max():%Y-%m}, "
              f"by source {df.groupby('source').size().to_dict()}, "
              f"below_min months: {int(df['below_min_flag'].sum())}")
else:
    print("USE_UNION_VARIANT is False: no union variants written.")

## 5. Limitations

- **Domain shift of InflaBERT.** The model was trained on FNSPID *summaries*
  from 2010–2019; here it scores *headlines*, and Alpaca extends to 2026.
- **Label skew.** About 70% of headlines are labeled "up", so the index moves
  mostly through the up/down mix rather than balanced classes.
- **Wire-style prefixes.** Reuters titles carry prefixes ("UPDATE 2-",
  "TREASURIES-", "BRIEF-") that Benzinga titles do not, so the two sources do
  not present text the same way to the model.
- **Mostly non-US inflation news.** Many matched headlines are about inflation
  in other countries (India, euro zone, Brazil, China…), not US CPI.
- **Term concentration.** The index is dominated by "fed" and "inflation"
  headlines.
- **Noisy early Alpaca months.** 2015–2017 Alpaca months have fewer articles,
  so their monthly means are noisier.

## Before committing: clear outputs

Run from the repo root so no outputs (or headline-derived content) are committed:

```bash
jupyter nbconvert --clear-output --inplace notebooks/model1/01_news_index.ipynb
```